# Case TechShop - E-commerce Analytics

Notebook principal do case. Cada questão segue o padrão `[MD explicação] -> [CODE] -> [MD análise]`, exceto Q6 (markdown-only).

# Bloco 1: Setup

Imports centralizados, carga do dataset e configuração de caminhos.

In [16]:
import pandas as pd
import numpy as np
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

RAW_PATH = Path('../data/raw/ecommerce_vendas.csv')
INTERIM_DIR = Path('../data/interim')
SQL_DIR = Path('../sql')

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

print('Bibliotecas carregadas!')

Bibliotecas carregadas!


# Bloco 2: Q1 — Diagnóstico de Qualidade

Mapeamento de problemas de qualidade do dataset antes de qualquer tratamento.

Checklist coberto: shape e tipos, nulos por coluna, missing disfarçado, duplicatas, domínios categóricos, resumo estatístico numérico, outliers e inconsistências cruzadas.

In [ ]:
df = pd.read_csv(RAW_PATH)

# --- 1. Shape e tipos: verificar se o dataset tem as 11 colunas esperadas e se os tipos estão corretos ---
print("=== Shape ===")
print(df.shape)
print("\n=== Tipos ===")
print(df.dtypes)
print("\n=== Amostra ===")
display(df.head())

# --- 2. Nulos: identificar colunas com ausências que impactam volume, receita e análise geográfica ---
print("\n=== Nulos absolutos e % ===")
nulos = df.isnull().sum()
percentual_nulos = (nulos / len(df) * 100).round(2)
display(pd.DataFrame({'nulos': nulos, 'percentual_%': percentual_nulos})[nulos > 0])

# --- 3. Missing disfarçado: capturar strings vazias que o pandas não converte automaticamente em NaN ---
print("\n=== Missing disfarçado ===")
colunas_texto = df.select_dtypes(include='object').columns
strings_vazias = [(col, (df[col].fillna('').str.strip() == '').sum())
                  for col in colunas_texto
                  if (df[col].fillna('').str.strip() == '').sum() > 0]
if strings_vazias:
    print('\n'.join(f'  {col}: {n}' for col, n in strings_vazias))
    print('  (idêntico aos NaN da seção 2 — sem ausências ocultas adicionais)')
else:
    print('  nenhum')

# --- 4. Duplicatas: confirmar se pedido_id é chave primária confiável antes de qualquer agregação ---
print("\n=== Duplicatas em pedido_id ===")
linhas_duplicadas = df[df.duplicated('pedido_id', keep=False)]
print(f"  Linhas duplicadas: {len(linhas_duplicadas)}  |  pedido_ids afetados: {linhas_duplicadas['pedido_id'].nunique()}")
display(linhas_duplicadas.sort_values('pedido_id'))

# --- 5. Domínios: detectar valores fora do padrão em status, categoria e uf que fragmentariam agrupamentos ---
print("\n=== Distribuição: status ===")
display(df['status'].value_counts(dropna=False))

print("\n=== Distribuição: categoria ===")
display(df['categoria'].value_counts(dropna=False))

print("\n=== UFs presentes ===")
UFS_VALIDAS = {
    'AC','AL','AP','AM','BA','CE','DF','ES','GO','MA','MT','MS','MG',
    'PA','PB','PR','PE','PI','RJ','RN','RS','RO','RR','SC','SP','SE','TO'
}
ufs_presentes = set(df['uf'].dropna().unique())
ufs_invalidas = ufs_presentes - UFS_VALIDAS
print(f"  Únicas (excl. nulos): {sorted(ufs_presentes)}")
print(f"  Fora do domínio: {ufs_invalidas if ufs_invalidas else 'nenhuma'}")

# --- 6. Estatísticas: obter range e distribuição das colunas numéricas como base para o diagnóstico de outliers ---
print("\n=== Resumo estatístico ===")
display(df[['qtd', 'valor_unit', 'desconto_%', 'avaliacao']].describe())

# --- 7a. Validação de domínio: checar violações de regras de negócio com limites fixos conhecidos ---
# Método: limiar fixo. Aplicável a colunas com domínio bem definido (qtd, desconto_%, avaliacao).
# Não detecta outliers estatísticos em valor_unit, que não tem limite superior de negócio definido.
print("\n=== Validação de domínio (regras de negócio) ===")
print(f"  qtd <= 0: {(df['qtd'] <= 0).sum()}")
print(f"  qtd não-inteira: {((df['qtd'].dropna() % 1) != 0).sum()}")
print(f"  valor_unit <= 0: {(df['valor_unit'] <= 0).sum()}")
print(f"  desconto_% < 0: {(df['desconto_%'] < 0).sum()}")
print(f"  desconto_% > 100: {(df['desconto_%'] > 100).sum()}")
print(f"  avaliacao fora de [1,5]: {((df['avaliacao'] < 1) | (df['avaliacao'] > 5)).sum()}")

# --- 7b. Outliers estatísticos em valor_unit: IQR, robusto a distribuições assimétricas ---
q1_valor = df['valor_unit'].quantile(0.25)
q3_valor = df['valor_unit'].quantile(0.75)
iqr_valor = q3_valor - q1_valor
limite_inferior_valor = q1_valor - 1.5 * iqr_valor  # IQR preferível ao z-score: distribuição assimétrica (mediana R$259 << média R$484) viola normalidade
limite_superior_valor = q3_valor + 1.5 * iqr_valor  # dois limites: inferior (Q1 - 1,5×IQR) e superior (Q3 + 1,5×IQR)

registros_acima_limite = df[df['valor_unit'] > limite_superior_valor]

print(f"\n=== Outliers estatísticos: valor_unit (método IQR) ===")
print(f"  Q1: R$ {q1_valor:.2f}  |  Q3: R$ {q3_valor:.2f}  |  IQR: R$ {iqr_valor:.2f}")
print(f"  Limite inferior (Q1 - 1,5 × IQR): R$ {limite_inferior_valor:.2f}")
print(f"  Limite superior (Q3 + 1,5 × IQR): R$ {limite_superior_valor:.2f}")
print(f"  Registros abaixo do limite inferior: {(df['valor_unit'] < limite_inferior_valor).sum()}"
      f"  — limite negativo (R$ {limite_inferior_valor:.2f}); nenhum preço pode assumir valor abaixo de zero")
print(f"  Registros acima do limite superior: {len(registros_acima_limite)}")
if not registros_acima_limite.empty:
    print(f"  Produtos envolvidos (acima do limite):")
    display(
        registros_acima_limite
        .groupby(['produto', 'categoria'])['valor_unit']
        .agg(pedidos='count', preco_unitario='first')
        .sort_values('preco_unitario', ascending=False)
        .reset_index()
    )

# --- 8. Cruzamento avaliacao x status: verificar se apenas pedidos entregues possuem avaliação preenchida ---
print("\n=== Avaliação por status ===")
avaliacao_por_status = df.groupby('status', observed=True).agg(
    total_pedidos=('pedido_id', 'count'),
    com_avaliacao=('avaliacao', 'count')
)
avaliacao_por_status['pct_avaliados'] = (
    avaliacao_por_status['com_avaliacao'] / avaliacao_por_status['total_pedidos'] * 100
).round(1)
display(avaliacao_por_status)
print("\n  Nulos de avaliacao por status (cross-check com seção 2 — total deve ser 241):")
display(df[df['avaliacao'].isna()]['status'].value_counts())

# --- 9. Datas: detectar formatos inconsistentes que o pandas converterá em NaT e excluirá da análise temporal ---
print("\n=== data_pedido ===")
df['data_pedido_parsed'] = pd.to_datetime(df['data_pedido'], errors='coerce')
datas_invalidas = df['data_pedido_parsed'].isna().sum()

print(f"  Válidas: {len(df) - datas_invalidas}  |  NaT: {datas_invalidas}  |  Range: {df['data_pedido_parsed'].min().date()} a {df['data_pedido_parsed'].max().date()}")
print(f"\n  Registros com formato inválido (dd/mm/yyyy) — {datas_invalidas} no total:")
display(
    df.loc[df['data_pedido_parsed'].isna(), ['pedido_id', 'data_pedido']]
    .sort_values('pedido_id')
    .reset_index(drop=True)
)

df.drop(columns=['data_pedido_parsed'], inplace=True)

### Achados — Q1

O dataset tem **`1.205` linhas e `11` colunas** e é utilizável, mas contém `11` problemas em cinco categorias que, se ignorados, produzem distorções silenciosas em cálculos de receita, volume e satisfação do cliente.

---

#### Evidências-chave

**Campos em branco**

| Coluna | Ausências | % | Impacto downstream |
|---|---|---|---|
| `avaliacao` | `241` | `20,00%` | Todos em `cancelado` (`138`) e `em_transito` (`103`) — padrão esperado; análises de satisfação exigem filtro por `status` antes de calcular médias |
| `desconto_%` | `30` | `2,49%` | Esses pedidos ficam fora de qualquer cálculo de margem líquida |
| `cliente_id` | `25` | `2,07%` | Pedidos anônimos não entram em segmentação nem análise comportamental por cliente |
| `uf` | `15` | `1,24%` | Excluídos automaticamente de relatórios regionais; se concentrados em uma região, a análise geográfica estará sistematicamente distorcida |
| `qtd` | `12` | `1,00%` | Não contribuem para volume nem receita |

A verificação de missing disfarçado (strings vazias) retornou os mesmos `25` e `15` já capturados — sem ausências ocultas adicionais. O `desconto_%` não tem violações de domínio (`< 0`: `0`; `> 100`: `0`); o máximo observado é `20%`, sugerindo um teto de política de negócio — referência para definir o limite de validação em Q2.

**Padrão de avaliação por status:** `cancelado` (`138`) e `em_transito` (`103`) têm `0%` de avaliação; `entregue` (`872`) e `devolvido` (`82`) têm `100%`. O padrão é coerente: clientes avaliam ao receber. Os `964` pedidos com avaliação têm média de `3,92/5` e desvio padrão de `1,16` — baseline de satisfação antes de qualquer filtragem. Ressalva importante: os `82` devolvidos têm nota registrada antes da decisão de devolução; incluí-los em médias de satisfação infla o indicador sem refletir a experiência pós-venda.

> A tabela do output exibe `6` linhas em vez de `4` porque `Entregue` (`8`) e `Devolvido` (`2`) — variantes com inicial maiúscula de DI-008 — aparecem como grupos distintos de `entregue` e `devolvido`. Ambos têm `pct_avaliados = 100,0`; os `10` registros estão contabilizados no total de `1.205`.

**Pedidos duplicados:** `10` linhas correspondem a apenas `5` pedidos distintos — sem remoção, contagens e somatórios de receita estão inflados. Três pares são cópias byte-a-byte (pedidos `1135`, `1618`, `1906`). Dois têm agravante: pedido `1113` tem `cliente_id` divergente entre as cópias; pedido `1885` tem `categoria` divergente (`"Monitores"` vs `"monitores"`), acumulando duplicata e inconsistência de capitalização.

**Grafias inconsistentes:** `status` — `10` registros com inicial maiúscula (`Entregue`: `8`, `Devolvido`: `2`) escapam de filtros padrão por `"entregue"` e `"devolvido"`, subestimando receita e volumes entregues. `categoria` — `15` registros em lowercase distribuídos em seis categorias geram `12` grupos em vez dos `6` esperados, distorcendo participações de categoria e rankings de receita.

**Quantidades inválidas:** `5` registros com `qtd <= 0` (mínimo = `-1`), sem interpretação válida para o negócio. A distribuição de `qtd` confirma pedidos tipicamente unitários — mediana `1`, percentil `75` igual a `2`, máximo `5` — tornando esses desvios pontuais mas rastreáveis.

**Outliers estatísticos em `valor_unit` (método IQR):** `valor_unit` não tem teto por regra de negócio e sua distribuição é assimétrica (mediana `R$ 259` bem abaixo da média `R$ 484`, desvio padrão `R$ 652,84` — maior que a própria média), o que invalida o z-score. O IQR é robusto a assimetrias e define dois limites: inferior `R$ -576` (negativo — nenhum preço pode assumir valor abaixo de zero; `0` violações) e superior `R$ 1.304` (`118` registros acima, em `6` produtos legítimos de alto valor: Monitor 24" `R$ 8.990`, Headset 7.1 e HD Externo 2TB `R$ 3.490`, Monitor 32" 4K `R$ 2.499`, SSD 500GB `R$ 1.890`, Monitor 27" 144Hz `R$ 1.499`). Esses `118` registros revelam cauda longa de preço: poucas SKUs de alta performance concentram parcela desproporcional da receita, tornando a média por pedido um indicador pouco representativo da venda típica.

> A tabela do output exibe `8` linhas em vez de `6` porque Monitor 32" 4K e Monitor 27" 144Hz aparecem duas vezes cada: uma com `categoria = "Monitores"` e outra com `"monitores"` (DI-009). O `groupby(['produto', 'categoria'])` trata os dois valores de capitalização como grupos distintos.

**Datas mal formadas:** `20` registros usam o formato `dd/mm/yyyy` (ex.: pedido `2146` → `"11/02/2024"`) em vez do padrão `yyyy-mm-dd`. São descartados silenciosamente como `NaT` em qualquer análise temporal — relatórios mensais, sazonalidade, tendências. O range das datas válidas cobre `2024-01-01` a `2024-12-31`.

---

#### O que isso significa para o negócio

Usar o dataset sem tratamento produz distorções concretas em pelo menos quatro frentes:

1. **Receita total superestimada** — os `5` pedidos duplicados entram duas vezes em qualquer soma de faturamento
2. **Relatórios regionais incompletos** — `15` pedidos sem `uf` somem de qualquer mapa ou ranking por estado; se concentrados em uma região, a análise geográfica estará sistematicamente distorcida
3. **Satisfação artificialmente inflada** — os `82` devolvidos têm nota registrada antes da devolução; incluí-los em médias de NPS eleva o indicador sem refletir a experiência pós-venda
4. **Agrupamentos quebrados sem aviso** — inconsistências de capitalização em `status` e `categoria` criam grupos duplicados em relatórios gerenciais de forma silenciosa, sem mensagem de erro

---

#### Ressalvas

- O dataset cobre `13` dos `27` estados. Não é erro de qualidade, mas limita conclusões sobre cobertura geográfica da TechShop
- Os `118` registros acima do limite IQR são produtos legítimos — não devem ser removidos, mas precisam ser considerados ao interpretar médias de receita
- O diagnóstico descreve o dataset como exportado. Não é possível determinar se os problemas têm origem no sistema de gestão de pedidos ou na exportação

---

#### Próximo passo — Q2

Os `11` achados foram catalogados em `memory-bank/data-issues.md` (DI-001 a DI-011). Ordem de tratamento recomendada:

1. **DI-002** — remover duplicatas de `pedido_id`; para `1113`, preservar a cópia com `cliente_id` preenchido
2. **DI-003** — excluir ou corrigir `qtd <= 0`
3. **DI-008 / DI-009** — normalizar capitalização de `status` e `categoria` (pré-requisito para qualquer agrupamento)
4. **DI-010** — reprocessar `data_pedido` suportando ambos os formatos
5. **DI-001 / DI-006 / DI-007** — decidir estratégia para campos com ausências (excluir, manter com marcação ou imputar, conforme o que cada análise exigir)

# Bloco 3: Q2 - Tratamento

## Q2 - Tratamento e Justificativa

_A preencher via `/tratar`._

# Bloco 4: Q3 - SQL

## Q3 - Consultas SQL

_A preencher via `/consultar-sql`._

# Bloco 5: Q4 - Negócio

## Q4 - Analise de Negocio Aberta

_A preencher via `/analisar-negocio`._

# Bloco 6: Q5 - Debug

## Q5 - Encontre o Erro

_A preencher via `/encontrar-erro`._

# Bloco 7: Q6 - Modelagem Dimensional

## Q6 - Modelagem

_A preencher via `/modelar-dimensional` (markdown-only)._

# Bloco 8: Q7 - Insight

## Q7 - Pergunta Livre

_A preencher via `/insight-livre`._